## Term Paper - Winter Semester 25/26 Submission

#### **Research Topic: Is Attention All You Need in Low-Resource Settings? An empirical study of self attention integrated with BiLSTM vs Transformers for a low resource language; Swahili.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

pd.set_option("display.max_colwidth", None)

In [ ]:
holdout_train = pd.read_csv("Swahili_News_Train.csv")
holdout_test = pd.read_csv("Swahili_News_Test.csv")

print(f"Validate train column names: {holdout_train.columns},\nValidate test column names: {holdout_test.columns}")

In [ ]:
swahili_news = pd.read_csv("SwahiliNewsClassificationDataset.csv")
# swahili_news.head()

In [ ]:
# holdout_train.head(1)

In [ ]:
print(
    f"Training set has {swahili_news.shape[0]} rows while validation/developer set has {holdout_train.shape[0]} rows."
    )

In [ ]:
print(
    f"Categories in training but missing in validation: {set(swahili_news["category"].str.lower().unique()) - set(holdout_train["category"].str.lower().unique())}"
    )

In [ ]:
print(
    f"Categories in validation but missing in training: {set(holdout_train["category"].str.lower().unique()) - set(swahili_news["category"].str.lower().unique())}"
    )

In [ ]:
print(
    f"Categories in validation but missing in training: {set(holdout_train["category"].unique()) - set(swahili_news["category"].unique())}"
    )

In [ ]:
print(
    f"Categories is training data: {swahili_news["category"].str.lower().unique()}.\nCategories in validation set: {holdout_train["category"].str.lower().unique()}"
    )

In [ ]:
set(
    holdout_train["category"].str.lower().unique()
    ) & set(
        swahili_news["category"].str.lower().unique()
        )

In [ ]:
swahili_news["category"].value_counts()

In [ ]:
holdout_train["category"].value_counts()

In [ ]:
sns.countplot(
    x="category",
    data=swahili_news,
    color="#fdaa48",
    order=swahili_news["category"].value_counts().index
    )

In [ ]:
sns.countplot(
    x="category",
    data=holdout_train,
    color="#72d09c",
    order=holdout_train["category"].value_counts().index
    )

In [ ]:
holdout_train[holdout_train["content"] == str(swahili_news.loc[1, "content"].lower().rstrip().lstrip())]

In [ ]:
swahili_news.loc[23258, "content"].lower().rstrip().lstrip()

In [ ]:
# set(holdout_train["id"].unique().tolist()) - set(swahili_news["id"].unique().tolist())

In [ ]:
len(set(holdout_train["id"].unique().tolist()) & set(swahili_news["id"].unique().tolist()))

In [ ]:
# swahili_news[swahili_news["id"].isin(list(set(holdout_train["id"].unique().tolist()) & set(swahili_news["id"].unique().tolist())))]

In [ ]:
# holdout_train[holdout_train["id"].isin(list(set(holdout_train["id"].unique().tolist()) & set(swahili_news["id"].unique().tolist())))]

In [ ]:
# merge the datasets
merged_swahili_news = pd.concat([swahili_news, holdout_train])

In [ ]:
swahili_news.shape[0] + holdout_train.shape[0]

In [ ]:
# swahili_news[swahili_news["category"] == "kitaifa"].head()

In [ ]:
holdout_train[holdout_train["category"] == "kitaifa"].head()

In [ ]:
[news for news in swahili_news["content"].tolist() if "serikali imesema haitakuwa" in news]

In [ ]:
merged_swahili_news["category"].unique()

In [ ]:
merged_swahili_news[merged_swahili_news["content"].duplicated()].loc[0, "content"]

In [ ]:
# merged_swahili_news[merged_swahili_news["content"] == "serikali imesema haitakuwa tayari kuona amani na utulivu wa nchi inachezewa huku ikisisitiza uwepo wa umoja kati ya wananchi bila kujali tofauti ya imani, kabila au itikadi yoyote.hayo yalisemwa na naibu waziri wa mambo ya ndani ya nchi, hamad yussuf masauni wakati akifungua semina ya siku mbili iliyofanyika jijini dar es salaam ikiwahusisha viongozi wa taasisi za kiislamu, lengo ikiwa ni kuwakumbusha kuhubiri amani katika sehemu zao.naibu waziri amesema mwelekeo na malengo ya serikali ya awamu ya tano ni kukuza maendeleo katika sehemu mbalimbali nchini lengo ikiwa kuinua maisha ya wananchi na nchi kwa ujumla.“serikali hii imejidhatiti kuhakikisha maendeleo yanakuja kwa kasi na maendeleo hayawezi kuja ikiwa amani na utulivu haupo, sisi kama serikali tutahakikisha tunalinda amani iliyopo ili wananchi wapate kufanya shughuli za kiuchumi bila wasiwasi wowote,” amesema masauni.akizungumza wakati wa ufunguzi huo, shehe wa mkoa wa dar es salaam, alhaji alhad mussa salum aliihakikishia serikali kutokuwepo kwa mifarakano kati ya taasisi mbalimbali kama ilivyokuwepo awali huku akisisitiza kuendelea kwa umoja huo ili jamii ipate kuendelea.“sisi kama bakwata tunaihakikishia serikali uwepo wa umoja na ushirikiano baina ya baraza na taasisi zingine na tofauti zetu hazipelekei kukoseana au kuvunjiana heshima kwahiyo tunaomba serikali iamini uwepo wa maelewano mazuri tu kwa maendeleo ya nchi hii,” amesema shehe alhad.semina hiyo ya siku mbili imejumuisha viongozi wa taasisi 100 huku mada ya nafasi ya taasisi za kiislamu katika kuleta umoja na kuishi kwa amani itajadiliwa."]

In [ ]:
# tokenizer.tokenize(merged_swahili_news[merged_swahili_news["content"] == "serikali imesema haitakuwa tayari kuona amani na utulivu wa nchi inachezewa huku ikisisitiza uwepo wa umoja kati ya wananchi bila kujali tofauti ya imani, kabila au itikadi yoyote.hayo yalisemwa na naibu waziri wa mambo ya ndani ya nchi, hamad yussuf masauni wakati akifungua semina ya siku mbili iliyofanyika jijini dar es salaam ikiwahusisha viongozi wa taasisi za kiislamu, lengo ikiwa ni kuwakumbusha kuhubiri amani katika sehemu zao.naibu waziri amesema mwelekeo na malengo ya serikali ya awamu ya tano ni kukuza maendeleo katika sehemu mbalimbali nchini lengo ikiwa kuinua maisha ya wananchi na nchi kwa ujumla.“serikali hii imejidhatiti kuhakikisha maendeleo yanakuja kwa kasi na maendeleo hayawezi kuja ikiwa amani na utulivu haupo, sisi kama serikali tutahakikisha tunalinda amani iliyopo ili wananchi wapate kufanya shughuli za kiuchumi bila wasiwasi wowote,” amesema masauni.akizungumza wakati wa ufunguzi huo, shehe wa mkoa wa dar es salaam, alhaji alhad mussa salum aliihakikishia serikali kutokuwepo kwa mifarakano kati ya taasisi mbalimbali kama ilivyokuwepo awali huku akisisitiza kuendelea kwa umoja huo ili jamii ipate kuendelea.“sisi kama bakwata tunaihakikishia serikali uwepo wa umoja na ushirikiano baina ya baraza na taasisi zingine na tofauti zetu hazipelekei kukoseana au kuvunjiana heshima kwahiyo tunaomba serikali iamini uwepo wa maelewano mazuri tu kwa maendeleo ya nchi hii,” amesema shehe alhad.semina hiyo ya siku mbili imejumuisha viongozi wa taasisi 100 huku mada ya nafasi ya taasisi za kiislamu katika kuleta umoja na kuishi kwa amani itajadiliwa."]["content"])

In [ ]:
def text_preprocessing(text):
    text = text.lower().rstrip().lstrip()
    return re.sub(r'[.,\'\[\]]', '', text)

merged_swahili_news[["content", "category"]] = merged_swahili_news[["content", "category"]].map(text_preprocessing)
merged_swahili_news.head(1)

In [ ]:
merged_swahili_news.tail(1)

In [ ]:
merged_swahili_news[merged_swahili_news["content"].duplicated()].shape[0]

In [ ]:
merged_swahili_news[~merged_swahili_news["content"].duplicated()].shape[0]

In [ ]:
merged_swahili_news[merged_swahili_news["content"].duplicated()].shape[0] + merged_swahili_news[~merged_swahili_news["content"].duplicated()].shape[0]

In [ ]:
merged_swahili_news.shape

In [ ]:
print(f"{(merged_swahili_news[merged_swahili_news["content"].duplicated()].shape[0] / merged_swahili_news.shape[0]) * 100:.2f}% of the merged data set is duplicated.")

In [ ]:
merged_swahili_news = merged_swahili_news[~merged_swahili_news["content"].duplicated()]

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained("Benjamin-png/bert-tokenizer-swahili")

# model = AutoModelForTokenClassification.from_pretrained("castorini/afriberta_base")
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
masakhane_tokenizer = AutoTokenizer.from_pretrained("Davlan/afro-xlmr-base")
swah_tokenizer = AutoTokenizer.from_pretrained("Benjamin-png/swahili-mms-tts-finetuned")
# tokenizer.model_max_length = 512


inputs = tokenizer(
    # padding=True,
    stride=50,
    return_overflowing_tokens=True
)

def tokenize(text):
    encoding = tokenizer(
        text,
        max_length=512,
        truncation=True,
        stride=128,
        return_overflowing_tokens=True,
        padding="max_length"
    )
    return encoding["input_ids"]

In [ ]:
print("Tokenizing using xlm roberta..")

def tokenize(text):
    # tokens = tokenizer.tokenize(text)
    encoding = tokenizer(
                    text,
                    max_length=512,
                    truncation=True,
                    # stride=128,
                    return_overflowing_tokens=True,
                    padding="max_length"
                    )
    tokens = [tokenizer.convert_ids_to_tokens(ids) for ids in encoding["input_ids"]][0]
    token_ids = encoding["input_ids"][0]
    return tokens, token_ids

merged_swahili_news["tokenized"] = merged_swahili_news["content"].apply(lambda x: tokenize(x)[0])
merged_swahili_news["token_ids"] = merged_swahili_news["content"].apply(lambda x: tokenize(x)[1])

print("Now tokenizing using masakhane...")

# def masakhane_tokenize(text):
#     encoding = masakhane_tokenizer(
#                     text,
#                     max_length=512,
#                     truncation=True,
#                     stride=128,
#                     return_overflowing_tokens=True,
#                     padding="max_length"
#                     )
#     tokens = [masakhane_tokenizer.convert_ids_to_tokens(ids) for ids in encoding["input_ids"]][0]
#     token_ids = encoding["input_ids"]
#     return tokens, token_ids

# merged_swahili_news["masakhane_tokenized"] = merged_swahili_news["content"].apply(lambda x: masakhane_tokenize(x)[0])
# merged_swahili_news["masakhane_token_ids"] = merged_swahili_news["content"].apply(lambda x: masakhane_tokenize(x)[1])

# print("Working on tokenizing using SwahBERT...")

# def swah_tokenize(text):
#     encoding = swah_tokenizer(
#                     text,
#                     max_length=512,
#                     truncation=True,
#                     stride=128,
#                     return_overflowing_tokens=True,
#                     padding="max_length"
#                     )
#     tokens = [swah_tokenizer.convert_ids_to_tokens(ids) for ids in encoding["input_ids"]]
#     token_ids = encoding["input_ids"]
#     return tokens, token_ids

# merged_swahili_news["swah_tokenized"] = merged_swahili_news["content"].apply(lambda x: swah_tokenize(x)[0])
# merged_swahili_news["swah_token_ids"] = merged_swahili_news["content"].apply(lambda x: swah_tokenize(x)[1])

In [ ]:
merged_swahili_news.head(1)

In [ ]:
# merged_swahili_news.loc[0, ["token_ids", "masakhane_token_ids"]]

In [ ]:
merged_swahili_news["content"].loc[0]

In [ ]:
merged_swahili_news["token_length"] = merged_swahili_news["tokenized"].apply(lambda x: len(x))

pd.concat([merged_swahili_news[["token_length"]].sort_values(by="token_length").head(), merged_swahili_news[["token_length"]].sort_values(by="token_length").tail()])

In [ ]:
# merged_swahili_news["masakhane_token_length"] = merged_swahili_news["masakhane_tokenized"].apply(lambda x: len(x))

# pd.concat([merged_swahili_news[["masakhane_token_length"]].sort_values(by="masakhane_token_length").head(), merged_swahili_news[["masakhane_token_length"]].sort_values(by="masakhane_token_length").tail()])

In [ ]:
merged_swahili_news[merged_swahili_news["token_length"] == 0]

In [ ]:
merged_swahili_news[merged_swahili_news["token_length"] == 0]["id"].tolist()

In [ ]:
merged_swahili_news[merged_swahili_news["token_length"] < 10]

In [ ]:
# merged_swahili_news[merged_swahili_news["content"] == "serikali imesema haitakuwa tayari kuona amani na utulivu wa nchi inachezewa huku ikisisitiza uwepo wa umoja kati ya wananchi bila kujali tofauti ya imani, kabila au itikadi yoyote.hayo yalisemwa na naibu waziri wa mambo ya ndani ya nchi, hamad yussuf masauni wakati akifungua semina ya siku mbili iliyofanyika jijini dar es salaam ikiwahusisha viongozi wa taasisi za kiislamu, lengo ikiwa ni kuwakumbusha kuhubiri amani katika sehemu zao.naibu waziri amesema mwelekeo na malengo ya serikali ya awamu ya tano ni kukuza maendeleo katika sehemu mbalimbali nchini lengo ikiwa kuinua maisha ya wananchi na nchi kwa ujumla.“serikali hii imejidhatiti kuhakikisha maendeleo yanakuja kwa kasi na maendeleo hayawezi kuja ikiwa amani na utulivu haupo, sisi kama serikali tutahakikisha tunalinda amani iliyopo ili wananchi wapate kufanya shughuli za kiuchumi bila wasiwasi wowote,” amesema masauni.akizungumza wakati wa ufunguzi huo, shehe wa mkoa wa dar es salaam, alhaji alhad mussa salum aliihakikishia serikali kutokuwepo kwa mifarakano kati ya taasisi mbalimbali kama ilivyokuwepo awali huku akisisitiza kuendelea kwa umoja huo ili jamii ipate kuendelea.“sisi kama bakwata tunaihakikishia serikali uwepo wa umoja na ushirikiano baina ya baraza na taasisi zingine na tofauti zetu hazipelekei kukoseana au kuvunjiana heshima kwahiyo tunaomba serikali iamini uwepo wa maelewano mazuri tu kwa maendeleo ya nchi hii,” amesema shehe alhad.semina hiyo ya siku mbili imejumuisha viongozi wa taasisi 100 huku mada ya nafasi ya taasisi za kiislamu katika kuleta umoja na kuishi kwa amani itajadiliwa."]

In [ ]:
# merged_swahili_news.loc[5493, "tokenized"]

In [ ]:
merged_swahili_news.head(1)

In [ ]:
merged_swahili_news["token_length"] = merged_swahili_news["tokenized"].apply(lambda x: len(x))

In [ ]:
print(f"{round(merged_swahili_news[merged_swahili_news["token_length"] > 512].shape[0]/merged_swahili_news.shape[0] * 100, 2)}% of data has more than 512 tokens while {round(merged_swahili_news[merged_swahili_news["token_length"] <= 512].shape[0]/merged_swahili_news.shape[0] * 100, 2)}% has less than 512 tokens.")

In [ ]:
merged_swahili_news[merged_swahili_news["token_length"] > 512].shape[0]/merged_swahili_news.shape[0] * 100

In [ ]:
merged_swahili_news[merged_swahili_news["id"].isin(merged_swahili_news[merged_swahili_news["token_length"] > 512]["id"].unique().tolist())].head(1)

In [ ]:
merged_swahili_news_length = merged_swahili_news["token_ids"].str.len().max()
merged_swahili_news_length

In [ ]:
le = LabelEncoder()
merged_swahili_news["target"] = le.fit_transform(merged_swahili_news["category"])

merged_swahili_news.head(2)

In [ ]:
train_merged_swahili_news, test_merged_swahili_news = train_test_split(
                                                        merged_swahili_news,
                                                        train_size=.8,
                                                        random_state=42
                                                        )

train_merged_swahili_news, validation_merged_swahili_news = train_test_split(
                                                                train_merged_swahili_news,
                                                                train_size=0.9,
                                                                random_state=42,
                                                                stratify=train_merged_swahili_news["category"]
                                                                )

In [ ]:
print(f"Training data has {train_merged_swahili_news.shape[0]} rows, testing data has {test_merged_swahili_news.shape[0]} rows and the validation data has {validation_merged_swahili_news.shape[0]} rows.")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14,10))

sns.countplot(
    x="category",
    data=merged_swahili_news,
    color="#dbbaa9",
    order=merged_swahili_news["category"].value_counts().index,
    ax=ax[0,0]
    ).set_title("Swahili News")

sns.countplot(
    x="category",
    data=train_merged_swahili_news,
    color="#dba9ca",
    order=train_merged_swahili_news["category"].value_counts().index,
    ax=ax[0,1]
    ).set_title("Swahili News (Train Data)")

sns.countplot(
    x="category",
    data=test_merged_swahili_news,
    color="#fdaa48",
    order=test_merged_swahili_news["category"].value_counts().index,
    ax=ax[1,0]
    ).set_title("Swahili News (Test Data)")

sns.countplot(
    x="category",
    data=validation_merged_swahili_news,
    color="#72d09c",
    order=validation_merged_swahili_news["category"].value_counts().index,
    ax=ax[1,1]
    ).set_title("Swahili News (Validation Data)")

plt.tight_layout()

In [ ]:
merged_swahili_news.head(1)

In [ ]:
def collate(batch):
    samples, labels = zip(*batch)
    samples_padded = nn.utils.rnn.pad_sequence(samples, batch_first=True, padding_value=0)
    return samples_padded, torch.tensor(labels)

def inference_collate(batch):
    # samples = zip(*batch)
    samples_padded = nn.utils.rnn.pad_sequence(batch, batch_first=True, padding_value=0)
    return samples_padded

class SamplesDataset(Dataset):
    def __init__(self, data):
        self.samples = data["token_ids"].values
        self.targets = data["target"].values

    def __getitem__(self, index):
        sample = self.samples[index]
        target = self.targets[index]
        # return target
        return torch.tensor(sample, dtype=torch.long), torch.tensor(target, dtype=torch.long)
    
    def __len__(self):
        return len(self.samples)
    
class InferenceSamplesDataset(Dataset):
    def __init__(self, data):
        self.samples = data["token_ids"].values

    def __getitem__(self, index):
        sample = self.samples[index]
        return torch.tensor(sample, dtype=torch.long)
    
    def __len__(self):
        return len(self.samples)

train_data = SamplesDataset(train_merged_swahili_news)
validation_data = SamplesDataset(validation_merged_swahili_news)

train_loader = DataLoader(train_data, collate_fn=collate, shuffle=True, batch_size=64)
validation_loader = DataLoader(validation_data, collate_fn=collate, shuffle=True, batch_size=64)

In [ ]:
merged_swahili_news["target"].nunique()

In [ ]:
# Implementing LSTM

class SamplesLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, dimension):
        super().__init__()
        self.dimension = dimension
        self.vocab_size = vocab_size
        self.lstm = nn.LSTM(
            self.dimension,
            self.dimension,
            num_layers=2,
            # dropout=0.3,
            batch_first=True,
            bidirectional=True,
            )
        self.dropout = nn.Dropout(0.3)
        self.embedding = nn.Embedding(self.vocab_size, self.dimension, padding_idx=0)
        self.classification = nn.Linear(self.dimension * 2, 7)

    def forward(self,ins):
        # ins = ins.squeeze(1)
        embeddings = self.embedding(ins)
        # print(f"Before LSTM: {embeddings.shape}")
        output, (h_n, c_n) = self.lstm(embeddings)

        h = torch.cat((h_n[-2], h_n[-1]), dim=1)
        h = self.dropout(h)

        return self.classification(h)
        # return self.classification(h_n[-1])

In [ ]:
# max(max(chunk) for doc in merged_swahili_news["token_ids"] for chunk in doc) >= tokenizer.vocab_size

In [ ]:
merged_swahili_news.columns

In [ ]:
merged_swahili_news["tokenized"].head(1)

In [ ]:
vocab = {word for phrase in merged_swahili_news["tokenized"] for word in phrase}

In [ ]:
for sample, target in train_loader:
    print(target)
    # print(target.squeeze(1))
    print(f"Sample shape: {sample.shape}")
    print(f"Target shape: {target.shape}")
    print(target.ndim)
    break

In [ ]:
merged_swahili_news[merged_swahili_news["target"] == 4][["category", "target"]].head()

In [ ]:
len(train_loader)

In [ ]:
len(merged_swahili_news)

In [ ]:
for batch_X, batch_y in train_loader:
    print(batch_X.shape)
    print(batch_y.shape)
    break

In [ ]:
dimension = 512
epochs = 10
vocab_size = tokenizer.vocab_size

lossFct = nn.CrossEntropyLoss()
criterion = nn.CrossEntropyLoss()


lstm_model = SamplesLSTMClassifier(vocab_size, dimension)
optimizer = torch.optim.AdamW(lstm_model.parameters())

losses = []
val_losses = []

best_val_loss = float("inf")
patience = 3
counter = 0
for epoch in range(epochs):
    lossAbs = 0
    epoch_loss = 0
    num_batch = 0
    # counter = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}]")

    for sample, target in progress_bar:
        # print(f"Sample count: {counter}")
        outputs = lstm_model(sample)
        loss = lossFct(outputs, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        num_batch += 1
        # counter+=1
        progress_bar.set_postfix(loss=f"{epoch_loss/num_batch:.4f}")
    
    losses.append(epoch_loss/num_batch)

    lstm_model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for sample, target in validation_loader:
            outputs = lstm_model(sample)
            loss = criterion(outputs, target)
            val_loss += loss.item()

            preds = outputs.argmax(dim=1)
            correct += (preds == target).sum().item()
            total += target.size(0)

            all_predictions.append(preds)
            all_targets.append(target)

    val_loss /= len(validation_loader)
    val_acc = correct / total
    val_losses.append(val_loss)

    print(f"Epoch[{epoch+1}/{epochs}], Train Loss: {epoch_loss/num_batch:.4f}, Val Loss: {val_loss:.4f}")

    
    # if val_loss < best_val_loss:
    #     best_val_loss = val_loss
    #     counter = 0
    # else:
    #     counter += 1

    # if counter >= patience:
    #     print(f"Early stopping at epoch {epoch+1} and saving the model")
    #     torch.save({
    #         "model_state_dict": lstm_model.state_dict(),
    #         "vocab_size": vocab_size,
    #         "dimension": dimension}, "best_model.pt")
    #     break

    # print(f"Epoch[{epoch+1}/{epochs}], Loss: {epoch_loss/num_batch}")

In [ ]:
# plotting the training
fig, ax = plt.subplots(1, 2, figsize=(8, 5))
ax[0].plot(losses, color="#dba9b1")
ax[0].set(title="Training Loss (LSTM)", xlabel="Epoch", ylabel="Loss")
# ax[1].plot(val_losses, color="#a9b1db")
# ax[1].set(title="Validation Loss (LSTM)", xlabel="Epoch", ylabel="Loss")

plt.tight_layout()
plt.show()

In [ ]:
# Implementing self attention layer on LSTM

class SamplesLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, dimension):
        super().__init__()
        self.dimension = dimension
        self.vocab_size = vocab_size
        self.lstm = nn.LSTM(
            self.dimension,
            self.dimension,
            num_layers=2,
            # dropout=0.3,
            batch_first=True,
            bidirectional=True,
            )
        self.dropout = nn.Dropout(0.3)
        self.embedding = nn.Embedding(self.vocab_size, self.dimension, padding_idx=0)
        self.classification = nn.Linear(self.dimension * 2, 7)

    def forward(self,ins):
        # ins = ins.squeeze(1)
        embeddings = self.embedding(ins)
        # print(f"Before LSTM: {embeddings.shape}")
        output, (h_n, c_n) = self.lstm(embeddings)

        # attention scores
        attn_scores = self.attention(output)

        # [batch, seq_len]
        attn_scores = attn_scores.squeeze(-1)

        # normalize
        attn_weights = nn.functional.F.softmax(attn_scores, dim=1)

        # [batch, seq_len, 1]
        attn_weights = attn_weights.unsqueeze(-1)

        # weighted sum
        context = torch.sum(attn_weights * output, dim=1)

        # h = torch.cat((h_n[-2], h_n[-1]), dim=1)
        # h = self.dropout(h)

        return self.classification(context)
        # return self.classification(h_n[-1])